<a href="https://colab.research.google.com/github/MitraShabani/Merge-Order-Bias/blob/main/merge_order_stability_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai sentence-transformers matplotlib numpy

In [ ]:
import json
import re
from sentence_transformers import SentenceTransformer, util
import matplotlib.pyplot as plt

embedder = SentenceTransformer('all-MiniLM-L6-v2') # measures the angle between two of these coordinate vectors.
print("Embedding model loaded.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Loading the results
with open("/content/drive/MyDrive/merge-order-bias/forward_merge_results.json", "r", encoding="utf-8") as f:
    forward = json.load(f)

with open("/content/drive/MyDrive/merge-order-bias/backward_merge_results.json", "r", encoding="utf-8") as f:
    backward = json.load(f)

chapter_summaries = forward["chapter_summaries"]  # same for both
num_chapters = forward["num_chapters"]

print(f"Loaded {num_chapters} chapters for both directions.")

In [ ]:
# Extract each chapter's FIRST and LAST appearance text
def get_first_appearance_text(merge_log, chapter_num):
    """Return the full summary text from the step where `chapter_num` was first introduced."""
    for step in merge_log:
        if chapter_num in step["chapters_included"]:
            return step["summary"]
    return None


def get_last_appearance_text(merge_log, chapter_num):
    """Return the full summary text from the LAST step that includes `chapter_num`
    (in this design, that's simply the final step, since chapters are never dropped)."""
    for step in reversed(merge_log):
        if chapter_num in step["chapters_included"]:
            return step["summary"]
    return None


print("Helper functions ready.")

In [ ]:
# Compute drift: Compute drift per chapter, per direction

""" For each chapter: compare its original standalone summary to (a) the step where it was first introduced, and (b) the FINAL summary as a whole.
The gap between (a) and (b) tells us how much that chapter's presence/similarity degraded from its first appearance to the end of the merge process."""

def compute_drift(merge_log, num_chapters):
    results = []
    for i in range(num_chapters):
        chapter_num = i + 1
        first_text = get_first_appearance_text(merge_log, chapter_num)
        last_text = get_last_appearance_text(merge_log, chapter_num)

        emb_first = embedder.encode(first_text, convert_to_tensor=True)
        emb_last = embedder.encode(last_text, convert_to_tensor=True)

        stability = util.cos_sim(emb_first, emb_last).item()

        results.append({
            "chapter": chapter_num,
            "stability": stability  # HIGH = barely changed from first appearance to the end (dominates/persists)
                                     # LOW = changed a lot (got compressed/diluted/rewritten heavily)
        })
    return results


forward_drift = compute_drift(forward["merge_log"], num_chapters)
backward_drift = compute_drift(backward["merge_log"], num_chapters)

print("Forward stability (first-appearance vs. final):")
for d in forward_drift:
    print(f"  Chapter {d['chapter']}: {d['stability']:.3f}")

print("\nBackward stability (first-appearance vs. final):")
for d in backward_drift:
    print(f"  Chapter {d['chapter']}: {d['stability']:.3f}")


In [ ]:
# Plot: stability (first-appearance vs. final) by chapter position,, forward vs. backward

chapters = [d["chapter"] for d in forward_drift]
forward_stability = [d["stability"] for d in forward_drift]
backward_stability = [d["stability"] for d in backward_drift]

plt.figure(figsize=(10, 6))
plt.plot(chapters, forward_stability, marker='o', label='Forward merge', color='steelblue')
plt.plot(chapters, backward_stability, marker='s', label='Backward merge', color='firebrick')
plt.xlabel("Chapter number (book order)")
plt.ylabel("Stability: first-appearance vs. final-step similarity")
plt.title("Does a chapter's own treatment stay stable, based on when it was introduced?")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(chapters)
plt.show()